
#**Implementación de optimizador ADAM en Modelo de Regresión Lineal**


-Librerias necesarias

In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

-Comprobacion de correlación entre los datos

In [48]:
datos = {"x": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
"y": [5, 7, 8, 11, 12, 14, 15, 17, 20, 19, 22, 24, 25, 27, 29, 30, 34, 33, 36, 38, 39, 42, 41, 44, 47, 48, 50, 51, 54, 55]}

In [49]:
df = pd.DataFrame(datos)

In [50]:
df.head()

,x,y
0,1,5
1,2,7
2,3,8
3,4,11
4,5,12


In [51]:
#correlación
df.corr()

,x,y
x,1.000000,0.998786
y,0.998786,1.000000


#**ADAM:**

In [52]:
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
y = np.array([5, 7, 8, 11, 12, 14, 15, 17, 20, 19, 22, 24, 25, 27, 29, 30, 34, 33, 36, 38, 39, 42, 41, 44, 47, 48, 50, 51, 54, 55])

In [53]:
len(x)

30

In [54]:
global_min_error = np.inf
error_anterior = np.inf

-Iteraciones hasta encontrar los mejores beta:

In [55]:
def ADAM(x, y, ln, epoch, batch, beta1, beta2):

  b0 = 0
  b1 = 0
  epsilon = 1e-8
  n = len(x)
  momentum = 0
  magnitude = 0
  num_iter = 0

  mt0 = momentum
  mt1 = momentum
  vt0 = magnitude
  vt1 = magnitude

  for epoca in range(epoch):

    if epoca % 1000 == 0:
      print(f"Epoca: {epoca}")

    random_idx = np.random.permutation(len(x))  #reorganiza aleatoriamente los indices en cada epoca
    x_random = x[random_idx]
    y_random = y[random_idx]

    for i in range(0, n, batch):  #empieza desde el principio hasta el final dando pasos de tamaño batch

      num_iter += 1

      inicio = i
      fin = min(i + batch, n) #elige el minimo para evitar pasarnos del final de los datos en caso de que no coincida ej: n = 33 i+batch = 35
      len_batch = fin - inicio

      y_pred = (b0 + b1*x_random[inicio : fin])

      db0 = np.sum(-(2/len_batch) * (y_random[inicio : fin]-y_pred))
      db1= np.sum(-(2/len_batch) * (x_random[inicio : fin]) * (y_random[inicio : fin]-y_pred))

      mt0 = ((beta1*mt0) + ((1-beta1)*db0))
      mt0_hat = (mt0 / (1 - beta1**(num_iter)))
      mt1 = ((beta1*mt1) + ((1-beta1)*db1))
      mt1_hat = (mt1 / (1 - beta1**(num_iter)))

      vt0 = ((beta2*vt0) + ((1-beta2)*(db0)**2))
      vt0_hat = (vt0 / (1 - beta2**(num_iter)))
      vt1 = ((beta2*vt1) + ((1-beta2)*(db1)**2))
      vt1_hat = (vt1 / (1 - beta2**(num_iter)))

      b0 , b1 = b0 - ((ln*mt0_hat/(np.sqrt(vt0_hat) + epsilon)))  ,  b1 - ((ln*mt1_hat/(np.sqrt(vt1_hat) + epsilon)))

  return b0 , b1


In [56]:
b0, b1 = ADAM(x, y, 0.00005, 50000, 5, 0.9, 0.999)

Epoca: 0
Epoca: 1000
Epoca: 2000
Epoca: 3000
Epoca: 4000
Epoca: 5000
Epoca: 6000
Epoca: 7000
Epoca: 8000
Epoca: 9000
Epoca: 10000
Epoca: 11000
Epoca: 12000
Epoca: 13000
Epoca: 14000
Epoca: 15000
Epoca: 16000
Epoca: 17000
Epoca: 18000
Epoca: 19000
Epoca: 20000
Epoca: 21000
Epoca: 22000
Epoca: 23000
Epoca: 24000
Epoca: 25000
Epoca: 26000
Epoca: 27000
Epoca: 28000
Epoca: 29000
Epoca: 30000
Epoca: 31000
Epoca: 32000
Epoca: 33000
Epoca: 34000
Epoca: 35000
Epoca: 36000
Epoca: 37000
Epoca: 38000
Epoca: 39000
Epoca: 40000
Epoca: 41000
Epoca: 42000
Epoca: 43000
Epoca: 44000
Epoca: 45000
Epoca: 46000
Epoca: 47000
Epoca: 48000
Epoca: 49000


In [57]:
y_pred = b0 + b1 * x
mse = np.mean((y - y_pred)**2)
rmse = np.sqrt(mse)

print("b0:", b0)
print("b1:", b1)
print("MSE:", mse)
print("RMSE:", rmse)

b0: 3.26883086450494
b1: 1.7181574007072185
MSE: 0.5378718337229567
RMSE: 0.7333974595831081


#**-Comparación:**

-Solucion cerrada LinearRegression:

In [67]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(x.reshape(-1, 1), y)
b0_lin, b1_lin = lin.intercept_, lin.coef_[0]

In [66]:
print(b0_lin, b1_lin)

3.268965517241387 1.7181312569521685


-ADAM de sklearn:

In [68]:
from sklearn.neural_network import MLPRegressor

mlp = MLPRegressor(hidden_layer_sizes=(), activation="identity", solver="adam",
                   batch_size=5, learning_rate_init=0.00005, beta_1=0.9, beta_2=0.999,
                   max_iter=50000, tol=0, n_iter_no_change=10**9, alpha=0.0)
mlp.fit(x.reshape(-1, 1), y)
b0_mlp, b1_mlp = mlp.intercepts_[0][0], mlp.coefs_[0][0, 0]

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [69]:
print(b0_mlp, b1_mlp)

3.268778030239605 1.7181322203471008


#**Tabla comparativa:**

In [74]:
def mse(b0, b1): return np.mean((y - (b0 + b1 * x))**2)

pd.DataFrame({
    "Método": ["LinearRegression", "ADAM desde cero", "Adam sklearn"],
    "b0": [b0_lin, b0, b0_mlp],
    "b1": [b1_lin, b1, b1_mlp],
    "MSE": [mse(b0_lin, b1_lin), mse(b0, b1), mse(b0_mlp, b1_mlp)]})

,Método,b0,b1,MSE
0,LinearRegression,3.268966,1.718131,0.537872
1,ADAM desde cero,3.268831,1.718157,0.537872
2,Adam sklearn,3.268778,1.718132,0.537872
